# Assignment 9 — Full Part A Capstone Demonstration

This executed notebook integrates **all seven minimum viable product features**: wallet creation, on-ramp, signed stablecoin transfer, off-ramp, independent verification, a payment API, and a human-readable blockchain explorer.

The flow is a **local classroom simulation**: R1,000 deposit → 1,000 mZAR issued → 750 mZAR transferred and mined → R750 simulated redemption. It does not move real funds.

This notebook uses the existing modules from Parts A(a)–A(g) and exercises their actual HTTP endpoints. Values such as wallet addresses, transaction IDs and block hashes change on every run. No outputs have been manually fabricated.

## 1. Start a fresh payment platform and create wallets
The API listens only on `127.0.0.1`, using a free port chosen by the operating system. No external web services are used.

In [1]:
import copy
import json
import subprocess
import sys
import threading
from pathlib import Path
from urllib.error import HTTPError
from urllib.request import Request, urlopen

from api import create_server
from blockchain import create_signed_transfer
from demo import call
from redemption import create_signed_redemption
from verification import verify_exported_chain, lookup_verified_transaction
from wallet import Wallet

server = create_server(port=0)
server_thread = threading.Thread(target=server.serve_forever, daemon=True)
server_thread.start()
base_url = f"http://127.0.0.1:{server.server_address[1]}"

sender = Wallet("Sender")
recipient = Wallet("Recipient")

print("Payment API:", base_url)
print("Sender address:", sender.address)
print("Recipient address:", recipient.address)
print("Reserve Agent is managed by the local payment API")
assert sender.address != recipient.address

Payment API: http://127.0.0.1:37579
Sender address: 8892057c749129c384775a37004b89bb8a4ba52a
Recipient address: 7814e266ff75ec48dc183168e9ff7dbc2c8f33e6
Reserve Agent is managed by the local payment API


## 2. On-ramp — simulated R1,000 deposit
At the stated **1:1 reserve rule**, the reserve ledger records the deposit and credits the Sender with 1,000 mZAR. The reserve event is recorded in the separate simulated ledger, **not as a mined on-chain payment**.

In [2]:
status, deposit = call(base_url, "POST", "/onramp", {"address": sender.address, "amount": 1000})
assert status == 201
assert deposit["event"]["minted_mzar"] == 1000
assert deposit["state"]["vault_reserves_zar"] == 1000
assert deposit["state"]["circulating_mzar"] == 1000
assert deposit["state"]["reserve_invariant_ok"]
print("HTTP:", status)
print(json.dumps(deposit["event"], indent=2))
print("Sender balance:", deposit["state"]["balances_mzar"][sender.address], "mZAR")
print("Reserve:", deposit["state"]["vault_reserves_zar"], "ZAR")

HTTP: 201
{
  "event_id": "68447b30d0ce42d7891fe15d0dcbe7c9",
  "type": "ON_RAMP",
  "timestamp_utc": "2026-10-12T22:34:50.192830+00:00",
  "sender_address": "8892057c749129c384775a37004b89bb8a4ba52a",
  "reserve_agent_address": "460eed9ea10ff027ba56ea45c4c433a6470c6c4f",
  "local_currency": "ZAR",
  "deposited_local": 1000,
  "stablecoin": "mZAR",
  "minted_mzar": 1000,
  "conversion_rate": "1:1"
}
Sender balance: 1000 mZAR
Reserve: 1000 ZAR


## 3. Negative cases — reject forged signature and overspending
A signed request becomes invalid if its amount is changed after signing. Requests for more than the available 1,000 mZAR must also be rejected.

In [3]:
forged = create_signed_transfer(sender, recipient.address, 750)
forged["amount"] = 751
status_bad_sig, invalid_signature = call(base_url, "POST", "/transactions/new", forged)

overdraft = create_signed_transfer(sender, recipient.address, 1001)
status_overdraft, insufficient_funds = call(base_url, "POST", "/transactions/new", overdraft)

status, unaffected = call(base_url, "GET", "/state")
assert status_bad_sig == 400
assert status_overdraft == 400
assert unaffected["pending_transactions"] == 0
assert unaffected["balances_mzar"][sender.address] == 1000

print("Tampered signature:", status_bad_sig, invalid_signature)
print("Insufficient funds:", status_overdraft, insufficient_funds)
print("Sender balance after invalid attempts:", unaffected["balances_mzar"][sender.address])

Tampered signature: 400 {'error': 'invalid transaction signature or structure'}
Insufficient funds: 400 {'error': 'insufficient mZAR balance'}
Sender balance after invalid attempts: 1000


## 4. Create and submit the valid signed 750 mZAR transfer
Submitting a valid transaction places it in the pending pool. **Balances do not move until mining succeeds.**

In [4]:
transfer = create_signed_transfer(sender, recipient.address, 750)
status, queued = call(base_url, "POST", "/transactions/new", transfer)
assert status == 202
assert queued["transaction_id"] == transfer["id"]

_, waiting = call(base_url, "GET", "/state")
status_before_mine, _ = call(base_url, "GET", "/verify/" + transfer["id"])
assert waiting["pending_transactions"] == 1
assert waiting["balances_mzar"][sender.address] == 1000
assert status_before_mine == 404

print("HTTP:", status)
print("Transaction ID:", transfer["id"])
print("Pending count:", waiting["pending_transactions"])
print("Sender balance before mining:", waiting["balances_mzar"][sender.address])
print("Verify before mining: HTTP", status_before_mine)

HTTP: 202
Transaction ID: cffc98a60ee946a8a5a386063510c0f8
Pending count: 1
Sender balance before mining: 1000
Verify before mining: HTTP 404


## 5. Mine the transaction and verify inclusion
A successful demo block uses proof-of-work difficulty **2 hexadecimal leading zeros**. Chain verification and signature verification are checked independently using the serialised chain.

In [5]:
status, mined = call(base_url, "POST", "/mine")
assert status == 201 and mined["chain_valid"]
block = mined["block"]
assert block["hash"].startswith("00")

status, onchain = call(base_url, "GET", "/chain")
assert status == 200 and onchain["length"] == 2
independent_audit = verify_exported_chain(onchain["chain"], onchain["difficulty"])
assert independent_audit["valid"]

status, inclusion = call(base_url, "GET", "/verify/" + transfer["id"])
assert status == 200
assert inclusion["included"] and inclusion["signature_valid"]
assert inclusion["block_hash"] == block["hash"]

status, after_mining = call(base_url, "GET", "/state")
assert after_mining["balances_mzar"][sender.address] == 250
assert after_mining["balances_mzar"][recipient.address] == 750
assert after_mining["vault_reserves_zar"] == 1000
assert after_mining["circulating_mzar"] == 1000

print("Block index:", block["index"])
print("Block hash:", block["hash"])
print("Nonce:", block["nonce"])
print("Hash attempts:", mined["attempts"])
print("Independent chain verification:", independent_audit["valid"])
print("Transaction included:", inclusion["included"])
print("Sender:", after_mining["balances_mzar"][sender.address], "mZAR")
print("Recipient:", after_mining["balances_mzar"][recipient.address], "mZAR")

Block index: 1
Block hash: 003082ecc18b4a100adca44abfc0f5e7472b6de2eda2a047a50589f3d5aa5ab3
Nonce: 43
Hash attempts: 44
Independent chain verification: True
Transaction included: True
Sender: 250 mZAR
Recipient: 750 mZAR


## 6. Reject tampered chain data
This check changes the amount in an exported copy of the blockchain. It must fail verification; the original chain remains untouched.

In [6]:
tampered_chain = copy.deepcopy(onchain["chain"])
tampered_chain[1]["transactions"][0]["amount"] = 749
rejected_audit = verify_exported_chain(tampered_chain, onchain["difficulty"])
assert rejected_audit["valid"] is False
assert independent_audit["valid"] is True
print("Tampered chain accepted:", rejected_audit["valid"])
print("Verifier explanation:", rejected_audit["error"])

Tampered chain accepted: False
Verifier explanation: block 1: stored hash does not match block content


## 7. Off-ramp — Recipient redeems 750 mZAR
The Recipient signs a redemption request referring to the confirmed transfer. The platform simulates a local R750 payout, retires 750 mZAR from circulation and reduces simulated reserves by R750. This payout is a reserve-ledger receipt, not a real bank transfer.

In [7]:
redemption_request = create_signed_redemption(recipient, transfer["id"], 750)
status, redeemed = call(base_url, "POST", "/offramp", redemption_request)
assert status == 201
receipt = redeemed["receipt"]
assert receipt["paid_local"] == 750
assert receipt["payout_status"] == "SIMULATED"

status, repeated = call(base_url, "POST", "/offramp", redemption_request)
assert status == 400

status, end_state = call(base_url, "GET", "/state")
assert status == 200
assert end_state["balances_mzar"][sender.address] == 250
assert end_state["balances_mzar"][recipient.address] == 0
assert end_state["vault_reserves_zar"] == 250
assert end_state["circulating_mzar"] == 250
assert end_state["reserve_invariant_ok"]

print("Receipt ID:", receipt["receipt_id"])
print("Payout:", receipt["paid_local"], "ZAR (SIMULATED)")
print("Second redemption attempt: HTTP", status)
print("Sender balance:", end_state["balances_mzar"][sender.address], "mZAR")
print("Recipient balance:", end_state["balances_mzar"][recipient.address], "mZAR")
print("Remaining reserves:", end_state["vault_reserves_zar"], "ZAR")
print("Outstanding supply:", end_state["circulating_mzar"], "mZAR")

Receipt ID: c41946bb7d5e4598864f33d4b2096894
Payout: 750 ZAR (SIMULATED)
Second redemption attempt: HTTP 200
Sender balance: 250 mZAR
Recipient balance: 0 mZAR
Remaining reserves: 250 ZAR
Outstanding supply: 250 mZAR


## 8. Show the actual explorer and save verifiable evidence
The HTML below is returned from the running local API after the completed payment. Explorer pages and JSON snapshots are saved for the group's report.

In [8]:
from IPython.display import HTML, display

out_dir = Path("evidence")
out_dir.mkdir(exist_ok=True)

with urlopen(base_url + "/explorer", timeout=15) as response:
    explorer_html = response.read().decode("utf-8")
with urlopen(base_url + "/explorer/tx/" + transfer["id"], timeout=15) as response:
    detail_html = response.read().decode("utf-8")

assert transfer["id"] in explorer_html
assert transfer["id"] in detail_html
assert block["hash"] in detail_html

(out_dir / "notebook_explorer.html").write_text(explorer_html, encoding="utf-8")
(out_dir / "notebook_transaction_detail.html").write_text(detail_html, encoding="utf-8")

record = {
    "sender_address": sender.address,
    "recipient_address": recipient.address,
    "deposit_event": deposit["event"],
    "transaction_id": transfer["id"],
    "block": block,
    "chain": onchain,
    "independent_verification": independent_audit,
    "transaction_inclusion": inclusion,
    "redemption_receipt": receipt,
    "final_state": end_state,
    "rejected_tamper": rejected_audit,
}
(out_dir / "notebook_evidence.json").write_text(json.dumps(record, indent=2), encoding="utf-8")

print("Actual explorer HTML returned: HTTP 200")
print("Actual transaction detail returned: HTTP 200")
print("Saved files in:", out_dir.resolve())
display(HTML(explorer_html))

Actual explorer HTML returned: HTTP 200
Actual transaction detail returned: HTTP 200
Saved files in: /mnt/data/Capstone_Part_A_Integrated/evidence


## 9. Regression tests for every module
The included test files check the wallet, reserve, blockchain, redemption, independent verification, API, explorer and the full end-to-end flow. These are real tests run on the current source code.

In [9]:
result = subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-p", "test_*.py", "-q"],
    capture_output=True,
    text=True,
    timeout=120,
)
print(result.stdout + result.stderr)
assert result.returncode == 0, "One or more automated tests failed"
print("Regression suite: PASS")

----------------------------------------------------------------------
Ran 63 tests in 10.357s

OK

Regression suite: PASS


## 10. Stop the temporary notebook server
The saved HTML snapshots remain viewable offline. To open a live explorer again, run `python integrated_demo.py --keep-open` in a terminal.

In [10]:
server.shutdown()
server.server_close()
server_thread.join(timeout=5)
assert not server_thread.is_alive()
print("Local notebook server stopped.")
print("PART A END-TO-END DEMONSTRATION PASSED")

Local notebook server stopped.
PART A END-TO-END DEMONSTRATION PASSED
